In [1]:
# %% [markdown]
# # 📊 Customer Churn EDA + Preprocessing
# *Notebook: 01_eda.ipynb | Stage 1-3 Complete*
# 
# **Author**: Anuj Chaudhary  
# **Purpose**: Load data, validate, explore, preprocess, and prepare for modeling

# %% [code]
# INSTALL DEPENDENCIES (Colab only - skip if running locally with venv)
!pip install -q pandas==2.0.3 numpy==1.24.3 matplotlib==3.7.2 seaborn==0.13.0 plotly==5.17.0 scikit-learn==1.3.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydot 4.0.1 requires pyparsing>=3.1.0, but you have pyparsing 3.0.9 which is incompatible.


In [2]:
# %% [code]
# IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Plot settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
%matplotlib inline

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


In [4]:
# %% [code]
# DATA VALIDATION CHECKS
# 1. Target column exists
if 'Attrition_Flag' not in df.columns:
    raise ValueError("❌ Attrition_Flag column missing - wrong dataset?")

# 2. Handle empty strings in categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    empty_count = (df[col] == '').sum()
    if empty_count > 0:
        print(f"⚠️ {col}: {empty_count} empty strings → replacing with 'Unknown'")
        df[col] = df[col].replace('', 'Unknown')

# 3. Check for unexpected nulls in key features
key_features = ['Credit_Limit', 'Total_Trans_Ct', 'Months_Inactive_12_mon']
for col in key_features:
    if df[col].isnull().any():
        print(f"⚠️ {col}: {df[col].isnull().sum()} null values found")

print("✅ Data validation complete")

NameError: name 'df' is not defined

In [ ]:
# %% [code]
print("🔍 DATA QUALITY CHECK")
print(f"\n1. Shape: {df.shape}")

missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"\n2. Missing Values:\n{missing[missing > 0]}")
else:
    print(f"\n2. Missing Values: None ✓")

print(f"\n3. Target Distribution (Attrition_Flag):")
print(df['Attrition_Flag'].value_counts(normalize=True).map('{:.2%}'.format))

print(f"\n4. Feature Types:\n{df.dtypes.value_counts()}")

print(f"\n5. Numeric Summary (key columns):")
numeric_cols = ['Customer_Age', 'Credit_Limit', 'Total_Trans_Ct', 'Total_Revolving_Bal']
print(df[numeric_cols].describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
# %% [code]
# Convert target to binary for modeling
df["Churn"] = df["Attrition_Flag"].map({
    "Existing Customer": 0,
    "Attrited Customer": 1
})

# Verify conversion
print(f"✅ Target converted: {df['Churn'].value_counts().to_dict()}")
df[['Attrition_Flag', 'Churn']].head()

In [ ]:
# %% [code]
# Create output directory for plots
os.makedirs("output/plots", exist_ok=True)
print("✅ output/plots directory ready")

# Key behavioural columns for analysis
behavioural_cols = [
    "Total_Trans_Ct",          # Transaction count
    "Months_Inactive_12_mon",  # Inactivity
    "Contacts_Count_12_mon",   # Support contacts
    "Credit_Limit",
    "Total_Relationship_Count" # Products held
]

# Generate and save boxplots
for col in behavioural_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="Churn", y=col, palette="Set2")
    plt.title(f"{col} by Churn Status", fontsize=13, fontweight='bold')
    plt.xlabel("Churn (1=Yes, 0=No)")
    plt.tight_layout()
    
    # Save high-res for portfolio
    plt.savefig(f"output/plots/{col}_by_churn.png", dpi=300, bbox_inches='tight')
    plt.close()  # Free memory
    print(f"✅ Saved: output/plots/{col}_by_churn.png")

In [ ]:
# %% [code]
# Drop ID & artifact columns
cols_to_drop = ["CLIENTNUM", "Attrition_Flag"]  # Drop original target, keep binary 'Churn'
df_clean = df.drop(columns=cols_to_drop, errors="ignore")
print(f"✅ Dropped {len(cols_to_drop)} columns. Remaining: {df_clean.shape[1]} features")

In [ ]:
# %% [code]
# Encode categorical variables
cat_cols = ["Gender", "Education_Level", "Marital_Status", "Income_Category", "Card_Category"]

le_dict = {}  # Store encoders for later reference
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    le_dict[col] = le
    print(f"✅ Encoded '{col}': {dict(zip(le.classes_, range(len(le.classes_))))}")

print("\n✅ All categorical variables encoded")

In [ ]:
# %% [code]
# Separate features & target
X = df_clean.drop(columns=["Churn"])
y = df_clean["Churn"]

# Stratified split (preserves 16% churn ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"✅ Split complete:")
print(f"   Train: {X_train.shape[0]} rows | Churn rate: {y_train.mean():.2%}")
print(f"   Test:  {X_test.shape[0]} rows  | Churn rate: {y_test.mean():.2%}")

In [ ]:
# %% [code]
# Combine for saving with split indicator
train_df = X_train.copy()
train_df["Churn"] = y_train
train_df["split"] = "train"

test_df = X_test.copy()
test_df["Churn"] = y_test
test_df["split"] = "test"

full_processed = pd.concat([train_df, test_df], ignore_index=True)

# Save with versioning
os.makedirs("data/processed", exist_ok=True)
full_processed.to_csv("data/processed/customers_v2.csv", index=False)
print("✅ Saved: data/processed/customers_v2.csv")

In [ ]:
# %% [code]
# Critical validation checks
assert full_processed.isnull().sum().sum() == 0, "❌ Missing values detected!"
assert "Churn" in full_processed.columns, "❌ 'Churn' column missing!"
assert set(full_processed["Churn"].unique()) == {0, 1}, "❌ Target must be binary!"
assert "split" in full_processed.columns, "❌ 'split' column missing!"

print("🎉 All preprocessing validations passed!")